# Chapter 6: Linked Lists

This notebook introduces singly, doubly, and circular linked lists with runnable JavaScript examples, plus a practical media player implementation.

## 1. Why linked lists?

Linked lists store nodes connected by pointers instead of contiguous memory locations. This makes insertion/removal flexible without shifting all elements.

In [1]:
(() => {
  class LinkedListNode {
    constructor(data, next = null) {
      this.data = data;
      this.next = next;
    }
  }

  class LinkedList {
    #head = null;
    #size = 0;

    #isInvalidPosition(position) {
      return position < 0 || position > this.#size;
    }

    #elementToString(element) {
      if (typeof element === 'object' && element !== null) return JSON.stringify(element);
      return String(element);
    }

    append(data) {
      const newNode = new LinkedListNode(data);
      if (!this.#head) {
        this.#head = newNode;
      } else {
        let current = this.#head;
        while (current.next !== null) current = current.next;
        current.next = newNode;
      }
      this.#size++;
    }

    prepend(data) {
      const newNode = new LinkedListNode(data, this.#head);
      this.#head = newNode;
      this.#size++;
    }

    insert(data, position) {
      if (this.#isInvalidPosition(position)) return false;
      if (position === 0) {
        this.prepend(data);
        return true;
      }

      const newNode = new LinkedListNode(data);
      let current = this.#head;
      let previous = null;
      let index = 0;

      while (index++ < position) {
        previous = current;
        current = current.next;
      }

      newNode.next = current;
      previous.next = newNode;
      this.#size++;
      return true;
    }

    indexOf(data, compareFunction = (a, b) => a === b) {
      let current = this.#head;
      let index = 0;
      while (current) {
        if (compareFunction(current.data, data)) return index;
        index++;
        current = current.next;
      }
      return -1;
    }

    #removeFromHead() {
      const nodeToRemove = this.#head;
      this.#head = this.#head.next;
      this.#size--;
      return nodeToRemove.data;
    }

    #removeFromMiddleOrEnd(position) {
      let nodeToRemove = this.#head;
      let previous = null;
      for (let index = 0; index < position; index++) {
        previous = nodeToRemove;
        nodeToRemove = nodeToRemove.next;
      }
      previous.next = nodeToRemove.next;
      this.#size--;
      return nodeToRemove.data;
    }

    removeAt(position) {
      if (this.#size === 0) throw new RangeError('Cannot remove from an empty list.');
      if (position < 0 || position >= this.#size) throw new RangeError('Invalid position.');
      if (position === 0) return this.#removeFromHead();
      return this.#removeFromMiddleOrEnd(position);
    }

    remove(data, compareFunction = (a, b) => a === b) {
      const index = this.indexOf(data, compareFunction);
      if (index === -1) return null;
      return this.removeAt(index);
    }

    isEmpty() {
      return this.#size === 0;
    }

    get size() {
      return this.#size;
    }

    clear() {
      this.#head = null;
      this.#size = 0;
    }

    toString() {
      let current = this.#head;
      let output = '';
      while (current) {
        output += this.#elementToString(current.data);
        current = current.next;
        if (current) output += ', ';
      }
      return output;
    }
  }

  const list = new LinkedList();
  list.append('First');
  list.append('Second');
  list.prepend('Zero');
  list.insert('Middle', 2);
  console.log('list:', list.toString());
  console.log('indexOf(Second):', list.indexOf('Second'));
  console.log('removeAt(1):', list.removeAt(1));
  console.log('after removeAt:', list.toString());
})();

list: Zero, First, Middle, Second
indexOf(Second): 3
removeAt(1): First
after removeAt: Zero, Middle, Second


## 2. Doubly linked list

Each node stores both `next` and `previous` pointers, enabling bidirectional traversal and efficient operations at both ends.

In [ ]:
(() => {
  class DoublyLinkedListNode {
    constructor(data, next = null, previous = null) {
      this.data = data;
      this.next = next;
      this.previous = previous;
    }
  }

  class DoublyLinkedList {
    #head = null;
    #tail = null;
    #size = 0;

    append(data) {
      const newNode = new DoublyLinkedListNode(data);
      if (!this.#head) {
        this.#head = newNode;
        this.#tail = newNode;
      } else {
        newNode.previous = this.#tail;
        this.#tail.next = newNode;
        this.#tail = newNode;
      }
      this.#size++;
    }

    prepend(data) {
      const newNode = new DoublyLinkedListNode(data);
      if (!this.#head) {
        this.#head = newNode;
        this.#tail = newNode;
      } else {
        newNode.next = this.#head;
        this.#head.previous = newNode;
        this.#head = newNode;
      }
      this.#size++;
    }

    removeFirst() {
      if (!this.#head) return null;
      const nodeToRemove = this.#head;
      this.#head = nodeToRemove.next;
      if (this.#head) this.#head.previous = null;
      else this.#tail = null;
      this.#size--;
      return nodeToRemove.data;
    }

    removeLast() {
      if (!this.#tail) return null;
      const nodeToRemove = this.#tail;
      this.#tail = nodeToRemove.previous;
      if (this.#tail) this.#tail.next = null;
      else this.#head = null;
      this.#size--;
      return nodeToRemove.data;
    }

    toString() {
      let current = this.#head;
      const values = [];
      while (current) {
        values.push(String(current.data));
        current = current.next;
      }
      return values.join(' <-> ');
    }

    toStringReverse() {
      let current = this.#tail;
      const values = [];
      while (current) {
        values.push(String(current.data));
        current = current.previous;
      }
      return values.join(' <-> ');
    }
  }

  const dlist = new DoublyLinkedList();
  dlist.append('A');
  dlist.append('B');
  dlist.append('C');
  console.log(dlist.toString());
  console.log(dlist.toStringReverse());
  console.log('removeFirst:', dlist.removeFirst());
  console.log('removeLast:', dlist.removeLast());
  console.log('remaining:', dlist.toString());
})();

## 3. Circular linked list

In circular lists, the last node points back to the head. This supports cyclical traversal and round-robin style logic.

In [ ]:
(() => {
  class Node {
    constructor(data, next = null) {
      this.data = data;
      this.next = next;
    }
  }

  class CircularLinkedList {
    #head = null;
    #size = 0;
    #current = null;

    append(data) {
      const newNode = new Node(data);
      if (!this.#head) {
        this.#head = newNode;
        newNode.next = this.#head;
        this.#current = this.#head;
      } else {
        let current = this.#head;
        while (current.next !== this.#head) current = current.next;
        current.next = newNode;
        newNode.next = this.#head;
      }
      this.#size++;
    }

    getCurrent() {
      if (!this.#current && this.#head) this.#current = this.#head;
      return this.#current ? this.#current.data : null;
    }

    rotate(steps = 1) {
      if (!this.#head || this.#size === 0) return;
      if (!this.#current) this.#current = this.#head;
      const moves = Math.abs(steps) % this.#size;
      for (let i = 0; i < moves; i++) this.#current = this.#current.next;
    }

    traverse(rounds, callback) {
      if (!this.#head || rounds <= 0) return;
      let current = this.#head;
      const total = this.#size * rounds;
      for (let index = 0; index < total; index++) {
        callback(current.data, index);
        current = current.next;
      }
    }

    toString() {
      if (!this.#head) return 'Empty';
      const values = [];
      let current = this.#head;
      do {
        values.push(String(current.data));
        current = current.next;
      } while (current !== this.#head);
      return `${values.join(' -> ')} -> (circular)`;
    }
  }

  const clist = new CircularLinkedList();
  clist.append('A');
  clist.append('B');
  clist.append('C');
  console.log(clist.toString());
  console.log('current:', clist.getCurrent());
  clist.rotate(2);
  console.log('current after rotate(2):', clist.getCurrent());
  clist.traverse(1, (data, index) => console.log(`Position ${index}: ${data}`));
})();

## 4. TypeScript linked list (reference)

TypeScript adds generics and compile-time type safety while keeping method logic the same.

```typescript
// src/06-linked_lists/ts/linked-list.ts
class LinkedListNode<T> {
  constructor(public data: T, public next: LinkedListNode<T> | null = null) {}
}

class LinkedList<T> {
  private head: LinkedListNode<T> | null = null;
  private size: number = 0;

  append(data: T): void {}
  prepend(data: T): void {}
  removeAt(position: number): T | null { return null; }
  indexOf(data: T, compareFunction?: (a: T, b: T) => boolean): number { return -1; }

  isEmpty(): boolean {
    return this.size === 0;
  }

  get length(): number {
    return this.size;
  }
}

export default LinkedList;
```

## 5. Practical project: Media player with a doubly circular linked list

Songs are inserted in alphabetical order and playback can move next/previous with looping.

In [ ]:
(() => {
  class MediaPlayerSong {
    constructor(songTitle) {
      this.songTitle = songTitle;
      this.previous = null;
      this.next = null;
    }
  }

  class MediaPlayer {
    #firstSong = null;
    #lastSong = null;
    #size = 0;
    #playingSong = null;

    #compareSongs(songTitle1, songTitle2) {
      return songTitle1.localeCompare(songTitle2);
    }

    #findIndexOfSortedSong(newSongTitle) {
      let currentSong = this.#firstSong;
      let i = 0;
      for (; i < this.#size && currentSong; i++) {
        const currentSongTitle = currentSong.songTitle;
        if (this.#compareSongs(currentSongTitle, newSongTitle) >= 0) return i;
        currentSong = currentSong.next;
      }
      return i;
    }

    #insertEmptyPlaylist(newSong) {
      this.#firstSong = newSong;
      this.#lastSong = newSong;
      newSong.next = newSong;
      newSong.previous = newSong;
    }

    #insertAtBeginning(newSong) {
      newSong.next = this.#firstSong;
      newSong.previous = this.#lastSong;
      this.#firstSong.previous = newSong;
      this.#lastSong.next = newSong;
      this.#firstSong = newSong;
    }

    #insertAtEnd(newSong) {
      newSong.next = this.#firstSong;
      newSong.previous = this.#lastSong;
      this.#lastSong.next = newSong;
      this.#firstSong.previous = newSong;
      this.#lastSong = newSong;
    }

    #insertInMiddle(newSong, position) {
      let currentSong = this.#firstSong;
      for (let i = 0; i < position - 1; i++) currentSong = currentSong.next;

      newSong.next = currentSong.next;
      newSong.previous = currentSong;
      currentSong.next.previous = newSong;
      currentSong.next = newSong;
    }

    addSongByTitle(newSongTitle) {
      const newSong = new MediaPlayerSong(newSongTitle);
      if (this.#size === 0) {
        this.#insertEmptyPlaylist(newSong);
      } else {
        const position = this.#findIndexOfSortedSong(newSongTitle);
        if (position === 0) this.#insertAtBeginning(newSong);
        else if (position === this.#size) this.#insertAtEnd(newSong);
        else this.#insertInMiddle(newSong, position);
      }
      this.#size++;
    }

    play() {
      if (!this.#playingSong && this.#firstSong) this.#playingSong = this.#firstSong;
      if (this.#playingSong) console.log(`Now playing: ${this.#playingSong.songTitle}`);
    }

    next() {
      if (this.#playingSong) {
        this.#playingSong = this.#playingSong.next;
        this.play();
      }
    }

    previous() {
      if (this.#playingSong) {
        this.#playingSong = this.#playingSong.previous;
        this.play();
      }
    }

    getCurrentSong() {
      return this.#playingSong ? this.#playingSong.songTitle : null;
    }

    playlistToString() {
      if (!this.#firstSong) return 'Empty playlist';
      const songs = [];
      let current = this.#firstSong;
      for (let i = 0; i < this.#size; i++) {
        songs.push(current.songTitle);
        current = current.next;
      }
      return songs.join(' -> ');
    }
  }

  const player = new MediaPlayer();
  player.addSongByTitle('Yellow Submarine');
  player.addSongByTitle('Hey Jude');
  player.addSongByTitle('Let It Be');
  player.addSongByTitle('Come Together');

  console.log('playlist:', player.playlistToString());
  player.play();
  player.next();
  player.next();
  player.next();
  player.next();
  player.previous();
  console.log('current song:', player.getCurrentSong());
})();

## 6. Running chapter files outside this notebook

JavaScript:

```bash
node src/06-linked_lists/js/linked-list.js
node src/06-linked_lists/js/doubly-linked-list.js
node src/06-linked_lists/js/circular-linked-list.js
node src/06-linked_lists/js/media-player-example.js
```

TypeScript:

```bash
npx ts-node src/06-linked_lists/ts/linked-list.ts
npx ts-node src/06-linked_lists/ts/doubly-linked-list.ts
npx ts-node src/06-linked_lists/ts/circular-linked-list.ts
npx ts-node src/06-linked_lists/ts/media-player-example.ts
```

## 7. Summary

Linked lists provide flexible insert/remove behavior through pointer manipulation. You practiced singly, doubly, and circular variations, and applied them in a media player project with sorted insertion and cyclic playback.